In [ ]:
!pip install "pymongo"
!pip install fastavro
!pip install mysql-connector-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import uuid
import time
from datetime import datetime
from dotenv import load_dotenv

import pandas as pd
from pymongo import MongoClient
from sqlalchemy import create_engine
import fastavro
import logging

In [ ]:
import os

# Permite que el usuario defina la ruta, o usa un valor por defecto
ruta_base = os.getenv("RUTA_DATOS", "./datos/")

ruta_credencial = f"{ruta_base}credenciales/"
ruta_log = f"{ruta_base}log/"
ruta_destino = f"{ruta_base}add/avro/"
archivo_log = f"{ruta_log}logs_adquisicion.log"
os.makedirs(ruta_destino, exist_ok=True)

In [ ]:
#Logging (solo lectura y AVRO)
logger = logging.getLogger("pipeline_mongo")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

file_handler = logging.FileHandler(archivo_log)
formatter = logging.Formatter("%(asctime)s | %(message)s")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

In [ ]:
#Variables de ejecución
run_id = str(uuid.uuid4())
inicio_proceso = time.time()

LAST_PROCESSED_DATE = datetime(2025, 1, 1)

registros_extraidos = 0
registros_cargados = 0
estado_global = "exitoso"

In [ ]:
load_dotenv(f"{ruta_credencial}.env")

In [ ]:
#replica: escritura doble
mongoClient = MongoClient(os.getenv("MONGO_URI"))
mongoClient

In [ ]:
# metodo show - listar base de datos
mongoClient.list_database_names()

In [ ]:
# use database
#mongo db: tengo los documentos (colecciones)
db = mongoClient['datosabiertos']
db = mongoClient.datosabiertos
db

In [ ]:
#show tables (colecciones)
db.list_collection_names()

In [ ]:
#llaves: son la puerta para la condicion, permiten un estilo de "where"
print(db.presidencia2024.count_documents({}))
db.presidencia2024.find_one()
#puedo manipular datos con funciones o con tipos

In [ ]:
coleccion = 'alumnos_taller'
print(db[coleccion].count_documents({}))
db[coleccion].find_one()

In [ ]:
print(db[coleccion].count_documents({'curp23': {'$exists':1}}))
db[coleccion].find_one({'curp23': {'$exists':1}})

In [ ]:
from bson import ObjectId

print(ObjectId("54873e4a7591b0b48f65faf5").generation_time)
print(ObjectId("5f873e4a7591b0b48f65fd1c").generation_time)

In [ ]:
LAST_OBJECT_ID = ObjectId("54873e4a7591b0b48f65faf5")
coleccion = 'alumnos_taller'
collection = db[coleccion]
try:
  filtro = {'_id': {'$gt': LAST_OBJECT_ID}} #mayor que el elemento
  documentos = list(collection.find(filtro))
  df = pd.DataFrame(documentos)
  registros_extraidos = df.shape[0]
  if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)
    logger.info(f"LECTURA_MONGO | run_id={run_id} | registros={registros_extraidos}")
    print("Registros extraídos:", registros_extraidos)
except Exception as e:
  estado_global = "error"
  logger.error(f"LECTURA_MONGO | run_id={run_id} | error={str(e)}")
  print("Error en extracción:", e)

In [ ]:
df

In [ ]:
df.count()

# Análisis importante para clase

**El dataset tiene:**
*   66% con edad
*   33% con evaluaciones
*   12% con materias

Eso implica:
*   La estructura es altamente irregular (típico en NoSQL).
*   Muy buen ejemplo para explicar:
*   Esquema flexible
*   Sparsity
*   Necesidad de  Normalización antes de pasar a SQL

In [ ]:
df[
    df["edad"].notna() &
    df["evaluaciones"].notna() &
    df["materias"].notna()
  ]

In [ ]:
df_copy = df.copy()

In [ ]:
df["edad_anios"] = df["edad"].apply(lambda x: x.get("anios") if isinstance(x, dict) else None)
df.drop(columns=["edad"], inplace=True) # inplace: dentro del mismo elemento

df[
    df["edad_anios"].notna() &
    df["evaluaciones"].notna() &
    df["materias"].notna()
  ]

In [ ]:
#Crear diccionario de equivalencias
# mapeo para base de datos, normaliza distintas fuentes, mapeo de llave y valor
map_materias = {
    "ING": "INGLES",
    "MAT": "MATEMATICAS",
    "FIS": "FISICA",
    "ESP": "ESPAÑOL",
    "HIS": "HISTORIA",
    "HM": "HISTORIA",
    "QUI": "QUIMICA",
    "OPT": "ETICA"
}

In [ ]:
#duplicidad de datos que posteriormente tengo que arreglar
df = df.explode("evaluaciones") #puedo explotar con cualquier registro
df.count()

In [ ]:
#Filtrar por email
df[df["email"] == "arturo_pspzaldivar@hotmail.com"] #se desdoblo el objeto

In [ ]:
df #tipo left join sin evaluacion de las llaves

In [ ]:

df["materia_eval"] = df["evaluaciones"].apply(lambda x: x.get("materia") if isinstance(x, dict) else None)
df["calificacion"] = df["evaluaciones"].apply(lambda x: x.get("calificacion") if isinstance(x, dict) else None)
df["fecha_eval"] = df["evaluaciones"].apply(lambda x: x.get("fecha") if isinstance(x, dict) else None)

df.drop(columns=["evaluaciones"], inplace=True)
df[df["email"] == "arturo_pspzaldivar@hotmail.com"]

In [ ]:
#Funcion para Normalizar Materias
def normalizar_materia(row):
    codigo = row["materia_eval"]
    lista_materias = row.get("materias")

    # Si existe catálogo materias en el documento
    if isinstance(lista_materias, list):
        nombre = map_materias.get(codigo)

        # Si el nombre mapeado existe en lista materias, sobrevive
        if nombre in lista_materias:
            return nombre

    # Si no existe materias o no coincide, usar mapeo si existe
    return map_materias.get(codigo, codigo)

In [ ]:
df.rename(columns={"materia_final": "materia_eval"}, inplace=True)
df

In [ ]:
df["materia_final"] = df.apply(normalizar_materia, axis=1) #pasar el renglon, y de ahi va sobre los datos que se van a ocupar 0:columna 1:renglon

df[df["email"] == "arturo_pspzaldivar@hotmail.com"]

In [ ]:
df.drop(columns=["materia_eval"], inplace=True)
df.rename(columns={"materia_final": "materia", "clave_alu": "matricula"}, inplace=True)
df[df["email"] == "arturo_pspzaldivar@hotmail.com"]

In [ ]:
df["alumno"] = df["ap_paterno"] + " " + df["ap_materno"] + " " + df["nombre"]
df.drop(columns=["ap_paterno", "ap_materno", "nombre"], inplace=True)
df.loc[258]

In [ ]:
df.columns

In [ ]:
fecha_incremental = LAST_PROCESSED_DATE.strftime("%Y%m%d")
nombre_archivo = f"alumnos_mongo_{fecha_incremental}.avro"
archivo_avro = f"{ruta_destino}{nombre_archivo}"

schema = {
    "doc": "Alumnos Mongo",
    "name": "Alumno",
    "namespace": "mx.unam.fes",
    "type": "record",
    "fields": [ #arreglo de objetos
        {"name": col, "type": ["null", "string"], "default": None} #desdoblamiento de listas
        for col in df.columns
    ]
}

print(schema)

In [ ]:
try:
  with open(archivo_avro, "wb") as avro_file: #abre el objeto, lo mantiene abierto, escribe en binario (datos serealizados)
    fastavro.writer(avro_file, schema, df.astype(str).to_dict(orient="records"))
    logger.info(f"ESCRITURA_AVRO | run_id={run_id} | registros={registros_extraidos} | archivo={archivo_avro}")
    print("Registros cargados:", registros_extraidos)
except Exception as e:
  estado_global = "error"
  logger.error(f"ESCRITURA_AVRO | run_id={run_id} | error={str(e)}")
  print("Error en carga:", e)

In [ ]:
df.astype(str).to_dict(orient="records")

In [ ]:
# Destino (analítica)
engine_destino = create_engine(
    f"mysql+mysqlconnector://"
    f"{os.getenv('DST_DB_USER')}:"
    f"{os.getenv('DST_DB_PASSWORD')}@"
    f"{os.getenv('DST_DB_HOST')}:"
    f"{os.getenv('DST_DB_PORT')}/"
    f"{os.getenv('DST_DB_NAME')}"
)

In [ ]:
# GROUP académico
df_group = df.groupby(["matricula", "alumno"]).agg(
    promedio_calificacion=("calificacion", "mean"),
    n_evaluaciones=("calificacion", "count"),
    max_calificacion=("calificacion", "max"),
    min_calificacion=("calificacion", "min")
).reset_index()

df_group["promedio_calificacion"] = df_group["promedio_calificacion"].round(2)
df_group[df_group['n_evaluaciones'] > 0]

In [ ]:
# SELECT matricula, alumno SUM(calif) AS promedio_calificacion, COUNT(calif) from alumnos

autor = "AUTOR"
df_group["tpago"] = 0
df_group["npago"] = 0
df_group["autor"] = autor
df_group["fuente"] = "mongo"

df_group = df_group[df_group['n_evaluaciones'] > 0][
    [
        "autor",
        "matricula",
        "alumno",
        "tpago",
        "npago",
        "promedio_calificacion",
        "n_evaluaciones",
        "fuente"
    ]
]
df_group.count()

In [ ]:
#CARGA A TABLA DESTINO (SIN LOG)
tabla_destino = "alumnos_metricas_dw"

try:
    with engine_destino.begin() as connection:
        df_group.to_sql(
            tabla_destino,
            con=connection,
            if_exists="append",
            index=False
        )

    log_info = {
        "evento": "carga_destino",
        "tabla": tabla_destino,
        "registros_insertados": len(df_group),
        "estado": "exitoso",
        "timestamp": datetime.now().isoformat()
    }
    registros_cargados += len(df_group)

    logger.info(f"Proceso de carga: {json.dumps(log_info)}")
    print("Carga exitosa en destino")

except Exception as e:
    estado_global = "error"
    logger.error(f"Error en carga destino: {str(e)}")
    print("Error en carga destino:", e)


*  Data Lake
    * .avro
    * parquet
* Envio de info
    * mongo
    * mysql

In [ ]:
! ls -la  {ruta_destino}

In [ ]:
! cat {archivo_avro}

* **df.loc:** columna o renglón
* **df.iloc:** busca por el número de indice del df

In [ ]:
! cat {archivo_log}

In [ ]:
#Visualizar ruta y si el archivo .env esta en el directorio
print(ruta_credencial)
! ls -la  {ruta_credencial}

#Visualizar el contenido del archivo .env
! cat {ruta_credencial}.env